In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load the data

In [ ]:
df = pd.read_csv("../scrapping/hiru_news_raw.csv")

## Filter the data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.info()

In [ ]:
news_per_day = df["date"].value_counts().sort_index()
news_per_day.head()

In [ ]:
news_per_day.plot(kind="line", title="Number of News Articles per Day")

In [ ]:
sns.barplot(x=news_per_day.index, y=news_per_day.values)
plt.xticks(rotation=45)
plt.title("Number of News Articles per Day")
plt.show()

In [ ]:
count = df["date"].value_counts()
print(count)

In [ ]:
from collections import Counter

text = " ".join(df["title"] + " " + df["description"])
words = text.split()

Counter(words).most_common(20)

## Adding the lables for the news using the letters

In [ ]:
def label_news(text):
    """
    Enhanced Sinhala news classification with improved keyword coverage
    and case-insensitive matching
    """
    # Convert to lowercase for case-insensitive matching
    text_lower = text.lower()

    # Define expanded keyword sets for each category
    politics_keywords = [
        "????????", "??????", "????????????", "????????", "????",
        "??????", "???", "????", "????", "???????", "???????",
        "???????", "???", "?????????", "?????????", "????",
        "?????", "????", "?????", "??????"
    ]

    sports_keywords = [
        "????????", "bpl", "icc", "???????", "???", "???????",
        "?????????", "?????", "????????", "??? ??????",
        "????? ???? ????????", "???????", "?????", "???????",
        "?????? ???", "???????", "?????", "??????", "???",
        "?????????", "??? ??????", "????????"
    ]

    international_keywords = [
        "????????", "?????????????", "??????", "???????",
        "????", "???????", "???????????", "?????????",
        "????? ??????", "????", "???????", "???????",
        "???????", "?????", "??????????", "?????",
        "???", "?????? ??????", "?????", "???????"
    ]

    crime_keywords = [
        "??????????", "????????", "??????", "??????",
        "??????????", "??????", "?????", "?????", "??????",
        "???", "?????", "??????", "?????????", "?????",
        "?????????", "?????", "???????", "???????????",
        "????", "?????", "?????", "???????"
    ]

    weather_keywords = [
        "????", "?????", "??????", "???????", "??????",
        "???????", "????", "??????", "??", "?????? ?????",
        "????", "????", "?????? ???????", "???????",
        "????????", "????? ???????", "????", "????"
    ]

    business_keywords = [
        "??????", "????????", "???? ????? ???????",
        "?????", "??", "????", "????????", "?????",
        "?????", "??????", "?????", "?????", "???????",
        "??????", "??????", "?????", "????", "???????"
    ]

    health_keywords = [
        "??????", "?????", "??????", "???", "?????",
        "?????", "???", "?????????", "????", "??????",
        "?????", "?????? ??????????", "?????????",
        "?????????", "????????", "?????? ????"
    ]

    education_keywords = [
        "????????", "?????", "?????????????", "??????",
        "????", "?????", "?????", "?????????", "?????",
        "???????? ??????????", "??????", "??????"
    ]

    # Score-based classification for better accuracy
    category_scores = {
        "Politics": sum(1 for k in politics_keywords if k.lower() in text_lower),
        "Sports": sum(1 for k in sports_keywords if k.lower() in text_lower),
        "International": sum(1 for k in international_keywords if k.lower() in text_lower),
        "Crime": sum(1 for k in crime_keywords if k.lower() in text_lower),
        "Weather": sum(1 for k in weather_keywords if k.lower() in text_lower),
        "Business": sum(1 for k in business_keywords if k.lower() in text_lower),
        "Health": sum(1 for k in health_keywords if k.lower() in text_lower),
        "Education": sum(1 for k in education_keywords if k.lower() in text_lower)
    }

    # Find category with highest score
    max_score = max(category_scores.values())

    # Return category with highest score, or "Other" if no matches
    if max_score > 0:
        return max(category_scores.items(), key=lambda x: x[1])[0]
    return "Other"


# Apply the enhanced classification
df["category"] = (df["title"] + " " + df["description"]).apply(label_news)

# Display distribution
print("Category Distribution:")
print(df["category"].value_counts())
print("\n" + "="*50)
print(f"Total articles: {len(df)}")
print(f"Categorized: {len(df[df['category'] != 'Other'])}")
print(f"Uncategorized (Other): {len(df[df['category'] == 'Other'])}")
print(f"Coverage: {len(df[df['category'] != 'Other'])/len(df)*100:.1f}%")

In [ ]:
df.head()

In [ ]:
df.to_csv("filtered_hiru_news.csv", index=False, encoding="utf-8-sig")

# Train the model

In [ ]:
df["text"] = df["title"] + " " + df["description"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])
df[["category", "label"]].head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["text"])
y = df["label"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
def predict_category(title, description):
    text = title + " " + description
    vec = vectorizer.transform([text])
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]

predict_category(
    "???????? ??? ?????? ???? ??",
    "????? ???? ??????? ?? ????? ????????"
)

## Saving the test data

In [ ]:
import joblib

joblib.dump(model, "news_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Model files saved successfully")